In [ ]:
import requests
import pandas as pd

url = "https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&order=market_cap_desc&per_page=100&page=1&sparkline=false"
data = requests.get(url).json()
df = pd.DataFrame(data)
df.to_csv("crypto_top100.csv", index=False)
print("CSV saved!")

CSV saved!


In [ ]:
from google.colab import files

files.download("crypto_top100.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import requests # Ensure requests is imported for fetching data

# --- Code to generate historical.csv (copied from _zEy3pWMECWV) ---
# Fetch 90 days of Bitcoin prices
url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart?vs_currency=usd&days=90"
data = requests.get(url).json()

# Extract timestamp and price
prices = data['prices']
df_historical = pd.DataFrame(prices, columns=['timestamp', 'Price'])
df_historical['Date'] = pd.to_datetime(df_historical['timestamp'], unit='ms')
df_historical = df_historical[['Date', 'Price']]

# Save to CSV - this ensures historical.csv is created before being read
df_historical.to_csv("historical.csv", index=False)
print("historical.csv saved with", len(df_historical), "rows")
# --- End of historical.csv generation code ---

# Load historical Bitcoin data for time series forecasting
df = pd.read_csv('historical.csv')
df['Date'] = pd.to_datetime(df['Date'])

# Create a more granular time feature: total seconds since the first entry
df['TimeInSeconds'] = (df['Date'] - df['Date'].min()).dt.total_seconds()

# Add lagged price as a feature
df['Price_Lag1'] = df['Price'].shift(1)
# The MA7 feature is calculated but will not be used in the model yet to keep it simpler for iterative forecasting.
# df['MA7'] = df['Price'].rolling(window=7).mean()

# Drop rows with NaN values introduced by shift() for training
df_model = df.dropna().copy()

# Features: Time in seconds, and Price_Lag1; target: price
X = df_model[['TimeInSeconds', 'Price_Lag1']]
y = df_model['Price']

# Train/test split (80% train, 20% test)
split = int(0.8 * len(df_model))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict on test set
y_pred = model.predict(X_test)

# Calculate errors
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = model.score(X_test, y_test)

print(f"MAE: {mae}, RMSE: {rmse}, R2: {r2}")

# Predict next 30 days iteratively
last_time_in_seconds = df['TimeInSeconds'].max()
last_actual_price = df['Price'].iloc[-1]
current_price_for_prediction = last_actual_price
average_time_interval_seconds = df['TimeInSeconds'].diff().mean()

num_future_points = 30 * 24 # 30 days, assuming hourly data points
future_predictions = []
future_dates = []

# Generate future timestamps and predict prices iteratively
for i in range(1, num_future_points + 1):
    future_time_in_seconds_val = last_time_in_seconds + i * average_time_interval_seconds
    future_date_val = df['Date'].max() + pd.Timedelta(seconds=i * average_time_interval_seconds)

    # Create a DataFrame for prediction with the new features
    future_features = pd.DataFrame({
        'TimeInSeconds': [future_time_in_seconds_val],
        'Price_Lag1': [current_price_for_prediction]
    })
    predicted_price = model.predict(future_features)[0]
    future_predictions.append(predicted_price)
    future_dates.append(future_date_val)
    current_price_for_prediction = predicted_price # Use predicted price as lag for next step

# Create forecast CSV
forecast_df = pd.DataFrame({'Date': future_dates, 'Predicted_Price': future_predictions})
forecast_df.to_csv('forecast.csv', index=False)

historical.csv saved with 2167 rows
MAE: 179.95054676573574, RMSE: 261.4879080380835, R2: 0.9846994620688276


In [ ]:
import requests
import pandas as pd

# Fetch 90 days of Bitcoin prices
url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart?vs_currency=usd&days=90"
data = requests.get(url).json()

# Extract timestamp and price
prices = data['prices']
df = pd.DataFrame(prices, columns=['timestamp', 'Price'])
df['Date'] = pd.to_datetime(df['timestamp'], unit='ms')
df = df[['Date', 'Price']]

# Save to CSV
df.to_csv("historical.csv", index=False)
print("historical.csv saved with", len(df), "rows")

historical.csv saved with 2167 rows


In [ ]:
from google.colab import files

files.download("forecast.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download("historical.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your current flat forecast file
# Replace 'forecast.csv' with your actual filename
df_forecast = pd.read_csv('forecast.csv')

# Ensure your Date column is sorted chronologically
df_forecast['Date'] = pd.to_datetime(df_forecast['Date'])
df_forecast = df_forecast.sort_values('Date').reset_index(drop=True)

# 2. Define parameters for realistic volatility
# Assuming asset price is around 1K-1.1K based on your dashboard (Next30High = 1.08K)
# We will inject a daily variance of ~1.5% with a minor upward/downward drift
np.random.seed(42) # Keeps the generation consistent
n_days = len(df_forecast)
baseline_price = df_forecast['Predicted_Price'].iloc[0]

# Generate a random walk centered around your model's target baseline
daily_returns = np.random.normal(loc=0.0005, scale=0.015, size=n_days) # mean drift, daily volatility
price_multipliers = np.exp(np.cumsum(daily_returns))

# Scale the random walk so it neatly anchors to your model's predicted target valuation
scaled_predictions = baseline_price * (price_multipliers / price_multipliers.mean())

# 3. Replace the flat column with the dynamic trend
df_forecast['Predicted_Price'] = np.round(scaled_predictions, 2)

# 4. Save the updated data back to CSV
df_forecast.to_csv('forecast_updated.csv', index=False)
print("Successfully generated dynamic forecast trend!")

Successfully generated dynamic forecast trend!


In [ ]:
from google.colab import files

files.download("forecast.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>